# Experiment 4: Term Frequency and Named Entity Recognition

## 4.1 Toolkit-based TF & NER

In [9]:
import sys

# Clear a partially imported pandas package left by a previous failed run
for module_name in list(sys.modules):
    if module_name == 'pandas' or module_name.startswith('pandas.'):
        del sys.modules[module_name]

import spacy
import pandas as pd
from collections import Counter
from pathlib import Path

# Load spaCy English model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import spacy.cli
    spacy.cli.download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

# Read input data from the notebook folder or the workspace root
input_path = Path('4.1_4.2_input.txt')
if not input_path.exists():
    input_path = Path('Experiment4') / '4.1_4.2_input.txt'
text = input_path.read_text(encoding='utf-8')

# Process with spaCy
doc = nlp(text)

# Extract Term Frequency (Filtering out punctuation and whitespace)
words = [token.text.lower() for token in doc if not token.is_punct and not token.is_space]
word_freq = Counter(words)

# Identify top 10 most frequent terms
top_10_words = word_freq.most_common(10)
print('Top 10 most frequent terms (Toolkit):')
for term, freq in top_10_words:
    print(f'{term}: {freq}')

# Save to CSV
df_tf_toolkit = pd.DataFrame(word_freq.items(), columns=['Term', 'Frequency'])
df_tf_toolkit.to_csv('4.1_TF_Toolkit.csv', index=False)
print('\nTerm frequencies saved to 4.1_TF_Toolkit.csv')

# Named Entity Recognition (NER)
print('\nNamed Entities found (Top 10):')
entities = [(ent.text, ent.label_) for ent in doc.ents]
df_ner = pd.DataFrame(entities, columns=['Entity', 'Label'])
display(df_ner.head(10))

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Top 10 most frequent terms (Toolkit):
a: 88
and: 86
the: 75
of: 42
to: 42
in: 42
can: 42
is: 37
may: 30
word: 30

Term frequencies saved to 4.1_TF_Toolkit.csv

Named Entities found (Top 10):


,Entity,Label
0,Natural Language Processing and Artificial Int...,ORG
1,NLP,ORG
2,English,LANGUAGE
3,Hindi,GPE
4,Assamese,NORP
5,Bengali,NORP
6,Tamil,GPE
7,NLP,ORG
8,NLP,ORG
9,NLP,ORG


## 4.2 Native Term-Frequency Analysis (Without Toolkit)

In [10]:
import string
import csv
from pathlib import Path

# Read input data from the notebook folder or the workspace root
input_path = Path('4.1_4.2_input.txt')
if not input_path.exists():
    input_path = Path('Experiment4') / '4.1_4.2_input.txt'
text = input_path.read_text(encoding='utf-8')

# Preprocessing: lowercase and remove punctuation
text_lower = text.lower()
translator = str.maketrans('', '', string.punctuation)
clean_text = text_lower.translate(translator)

# Tokenize by splitting on whitespace
tokens = clean_text.split()

# Calculate Term Frequency natively
native_word_freq = {}
for token in tokens:
    native_word_freq[token] = native_word_freq.get(token, 0) + 1

# Identify top 10 most frequent terms
sorted_freq = sorted(native_word_freq.items(), key=lambda x: x[1], reverse=True)
top_10_native = sorted_freq[:10]

print('Top 10 most frequent terms (Native):')
for term, freq in top_10_native:
    print(f'{term}: {freq}')

# Save to CSV natively
with open('4.2_TF_Native.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Term', 'Frequency'])
    writer.writerows(native_word_freq.items())

print('\nTerm frequencies saved to 4.2_TF_Native.csv')

Top 10 most frequent terms (Native):
a: 90
and: 86
the: 75
to: 42
in: 42
can: 41
of: 40
is: 37
may: 30
word: 30

Term frequencies saved to 4.2_TF_Native.csv


## 4.3 Native TF-IDF Computation

In [11]:
import math
import string

# Given Documents
docs = [
    'Natural language processing is a field of artificial intelligence.',
    'Natural language processing helps computers understand human language.',
    'Machine learning is an important part of artificial intelligence.'
]

def preprocess(doc):
    doc = doc.lower()
    doc = doc.translate(str.maketrans('', '', string.punctuation))
    return doc.split()

# 1. Preprocess and Tokenize
tokenized_docs = [preprocess(doc) for doc in docs]

# 2. Calculate TF for each term
tf_docs = []
for tokens in tokenized_docs:
    tf = {}
    total_words = len(tokens)
    for word in tokens:
        tf[word] = tf.get(word, 0) + 1
    for word in tf:
        tf[word] = tf[word] / total_words  # Term Frequency formulation
    tf_docs.append(tf)

# 3. Calculate DF for every term
df = {}
for tokens in tokenized_docs:
    unique_terms = set(tokens)
    for term in unique_terms:
        df[term] = df.get(term, 0) + 1

# 4. Calculate IDF using logarithmic formula
total_docs = len(docs)
idf = {}
for term, freq in df.items():
    idf[term] = math.log(total_docs / freq)

# 5. Calculate TF-IDF for every term in every document
tfidf_docs = []
for tf in tf_docs:
    tfidf = {}
    for term, tf_value in tf.items():
        tfidf[term] = tf_value * idf[term]
    tfidf_docs.append(tfidf)

# 6. Display results
for i, doc_tfidf in enumerate(tfidf_docs):
    print(f'\n--- Document {i+1} ---')
    print(f'{"Term":<15} | {"TF":<8} | {"DF":<2} | {"IDF":<8} | {"TF-IDF"}')
    print('-' * 55)
    
    displayed = set()
    for term in tokenized_docs[i]:
        if term not in displayed:
            t_tf = tf_docs[i][term]
            t_df = df[term]
            t_idf = idf[term]
            t_tfidf = doc_tfidf[term]
            print(f'{term:<15} | {t_tf:<8.4f} | {t_df:<2} | {t_idf:<8.4f} | {t_tfidf:.4f}')
            displayed.add(term)
            
    # Identify the top 10 terms with the highest TF-IDF score
    top_10 = sorted(doc_tfidf.items(), key=lambda x: x[1], reverse=True)[:10]
    print('\nTop Terms (Highest TF-IDF):')
    for term, score in top_10:
        print(f'{term}: {score:.4f}')


--- Document 1 ---
Term            | TF       | DF | IDF      | TF-IDF
-------------------------------------------------------
natural         | 0.1111   | 2  | 0.4055   | 0.0451
language        | 0.1111   | 2  | 0.4055   | 0.0451
processing      | 0.1111   | 2  | 0.4055   | 0.0451
is              | 0.1111   | 2  | 0.4055   | 0.0451
a               | 0.1111   | 1  | 1.0986   | 0.1221
field           | 0.1111   | 1  | 1.0986   | 0.1221
of              | 0.1111   | 2  | 0.4055   | 0.0451
artificial      | 0.1111   | 2  | 0.4055   | 0.0451
intelligence    | 0.1111   | 2  | 0.4055   | 0.0451

Top Terms (Highest TF-IDF):
a: 0.1221
field: 0.1221
natural: 0.0451
language: 0.0451
processing: 0.0451
is: 0.0451
of: 0.0451
artificial: 0.0451
intelligence: 0.0451

--- Document 2 ---
Term            | TF       | DF | IDF      | TF-IDF
-------------------------------------------------------
natural         | 0.1250   | 2  | 0.4055   | 0.0507
language        | 0.2500   | 2  | 0.4055   | 0.1014
proce